# Vision-Guided Robotic Picking Lab

## Full Pipeline Notebook

This notebook walks through the complete pipeline:

1. **Image Capture** — Acquire images from the Basler ACE2 camera
2. **Inference** — Run a trained Roboflow segmentation model
3. **Calibration** — ArUco-based pixel-to-mm conversion
4. **Pick Logic** — Convert mask centroids to robot coordinates
5. **Robot Execution** — Pick-and-place with the Dobot Magician E6

### Before You Start
- Make sure your Basler camera is connected (or use `--simulate` for webcam)
- Have your Roboflow API key and project name ready
- Have an ArUco marker (DICT_4X4_50) printed and measured
- The Dobot E6 should be powered on and homed (for Session 4)

---
## 0. Setup and Imports

Install required packages if not already installed, then import everything we need.
33You will need to install pylon from baser webpage: https://www.baslerweb.com/en/downloads/software/?downloadCategory.values.label.data=pylon

and dobot software: https://www.dobot-robots.com/service/download-center?keyword=&products%5B%5D=688

In [ ]:

# Install required packages (uncomment if needed)
# numpy<2 is required — cv2 and matplotlib are compiled against NumPy 1.x and crash on 2.x
!pip install --user numpy pypylon opencv-python opencv-contrib-python roboflow matplotlib pandas

In [ ]:
# NumPy version check — run this BEFORE the imports cell
# If NumPy 2.x is detected, this cell downgrades it automatically.
# After running, RESTART THE KERNEL, then continue from the next cell.
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True
)
np_version = result.stdout.strip()
major = int(np_version.split(".")[0]) if np_version else 0

if major >= 4:
    print(f"NumPy {np_version} detected — downgrading to numpy<2 (required for cv2 + matplotlib)...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "numpy<2", "--quiet"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print("Downgrade complete.")
        print("\n*** RESTART THE KERNEL NOW (Kernel → Restart Kernel), then re-run from cell 1. ***")
    else:
        print("ERROR during downgrade:", r.stderr)
else:
    print(f"NumPy {np_version} — OK, no action needed.")

In [ ]:
import os
import sys
import time
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# Display images inline in the notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

# Add scripts folder to path so we can import our modules
SCRIPTS_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'scripts')
if os.path.isdir(SCRIPTS_DIR):
    sys.path.insert(0, SCRIPTS_DIR)
    print(f"Scripts directory: {SCRIPTS_DIR}")
else:
    # Try relative path
    SCRIPTS_DIR = os.path.abspath('../scripts')
    if os.path.isdir(SCRIPTS_DIR):
        sys.path.insert(0, SCRIPTS_DIR)
        print(f"Scripts directory: {SCRIPTS_DIR}")
    else:
        print("WARNING: scripts directory not found. Adjust SCRIPTS_DIR manually.")

print("Setup complete.")

---
## 1. Image Capture

We will capture images from the Basler ACE2 camera. If you don't have a Basler camera connected, set `USE_WEBCAM = True` to use a USB webcam instead.

In [ ]:
# ============================================================
# CONFIGURATION — Change these settings as needed
# ============================================================
USE_WEBCAM = False          # Set to False if using a Basler camera
OUTPUT_DIR = "captured_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Images will be saved to: {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
def capture_single_frame(use_webcam=USE_WEBCAM):
    """Capture a single frame from camera (no preview)."""
    if use_webcam:
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            raise RuntimeError("Cannot open webcam")
        ret, frame = cap.read()
        cap.release()
        if not ret:
            raise RuntimeError("Failed to read from webcam")
        return frame
    else:
        from pypylon import pylon
        tl_factory = pylon.TlFactory.GetInstance()
        devices = tl_factory.EnumerateDevices()
        if len(devices) == 0:
            raise RuntimeError("No Basler camera found")
        camera = pylon.InstantCamera(tl_factory.CreateDevice(devices[0]))
        camera.Open()
        camera.StartGrabbing(pylon.GrabStrategy_LatestImageOnly)
        converter = pylon.ImageFormatConverter()
        converter.OutputPixelFormat = pylon.PixelType_BGR8packed
        grab_result = camera.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
        image = converter.Convert(grab_result)
        frame = image.GetArray()
        grab_result.Release()
        camera.StopGrabbing()
        camera.Close()
        return frame


# ── Live preview ─────────────────────────────────────────────
# A live camera window opens. Adjust the camera position, then
# press  Q  to close the window and keep the last frame.
# Preview is scaled to 640x480; the saved image is full resolution.
# ─────────────────────────────────────────────────────────────
PREVIEW_W, PREVIEW_H = 640, 480

print("Live camera feed — adjust camera position, then press Q to capture.")

frame = None

if USE_WEBCAM:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam")
    while True:
        ret, f = cap.read()
        if not ret:
            break
        frame = f
        preview = cv2.resize(frame, (PREVIEW_W, PREVIEW_H))
        cv2.imshow("Camera Feed  |  Press Q to capture", preview)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()
else:
    from pypylon import pylon
    tl_factory = pylon.TlFactory.GetInstance()
    devices = tl_factory.EnumerateDevices()
    if len(devices) == 0:
        raise RuntimeError("No Basler camera found")
    camera = pylon.InstantCamera(tl_factory.CreateDevice(devices[0]))
    camera.Open()
    camera.StartGrabbing(pylon.GrabStrategy_LatestImageOnly)
    converter = pylon.ImageFormatConverter()
    converter.OutputPixelFormat = pylon.PixelType_BGR8packed
    while camera.IsGrabbing():
        grab_result = camera.RetrieveResult(100, pylon.TimeoutHandling_Return)
        if grab_result.GrabSucceeded():
            img = converter.Convert(grab_result)
            frame = img.GetArray()
            preview = cv2.resize(frame, (PREVIEW_W, PREVIEW_H))
            cv2.imshow("Camera Feed  |  Press Q to capture", preview)
        grab_result.Release()
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cv2.destroyAllWindows()
    camera.StopGrabbing()
    camera.Close()

if frame is None:
    print("ERROR: No frame captured.")
else:
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"img_{timestamp}.png"
    filepath = os.path.join(OUTPUT_DIR, filename)
    cv2.imwrite(filepath, frame)
    print(f"Saved: {filepath}")
    print(f"Image size: {frame.shape[1]} x {frame.shape[0]} px")

    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(f"Captured: {filename}")
    plt.axis('off')
    plt.show()


In [ ]:
# Save the captured frame
timestamp = time.strftime("%Y%m%d_%H%M%S")
filename = f"img_{timestamp}.png"
filepath = os.path.join(OUTPUT_DIR, filename)
cv2.imwrite(filepath, frame)
print(f"Saved: {filepath}")

In [ ]:
# Image harvesting
# Capture multiple images — run this cell repeatedly with different object arrangements
# Each run captures and saves one image

frame = capture_single_frame(use_webcam=USE_WEBCAM)
count = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png')])
filename = f"img_{count:04d}.png"
filepath = os.path.join(OUTPUT_DIR, filename)
cv2.imwrite(filepath, frame)

plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"Captured: {filename} (total: {count + 1} images)")
plt.axis('off')
plt.show()

In [ ]:
# Review all captured images in a grid
image_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png')])
print(f"Total captured images: {len(image_files)}")

if image_files:
    n_show = min(12, len(image_files))
    cols = 4
    rows = math.ceil(n_show / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for i in range(len(axes)):
        if i < n_show:
            img = cv2.imread(os.path.join(OUTPUT_DIR, image_files[i]))
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[i].set_title(image_files[i], fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No images found. Run the capture cells above first.")

---
## 2. Model Training

Training happens in the Roboflow web interface:
1. Go to [app.roboflow.com](https://app.roboflow.com)
2. Upload your captured images
3. Annotate objects with polygon masks
4. Train a segmentation model

Once training is complete, enter your API details below.

In [ ]:
# ============================================================
# TODO: Enter your Roboflow credentials
# ============================================================
ROBOFLOW_API_KEY = "6wF63p7WTrxTL3Og4vrO"      # <-- CHANGE THIS
ROBOFLOW_PROJECT = "markers-uorw8"      # <-- CHANGE THIS
ROBOFLOW_MODEL_VERSION = 2                  # <-- CHANGE if needed

# Confidence threshold for detections (0.0 to 1.0)
CONFIDENCE_THRESHOLD = 0.8                   # <-- TRY CHANGING THIS
# ============================================================

print(f"Project: {ROBOFLOW_PROJECT}")
print(f"Version: {ROBOFLOW_MODEL_VERSION}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

---
## 3. Inference

Run the trained model on images to detect and segment objects.

In [ ]:
from roboflow import Roboflow

# Connect to Roboflow and load model
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace().project(ROBOFLOW_PROJECT)
model = project.version(ROBOFLOW_MODEL_VERSION).model
print(f"Model loaded: {ROBOFLOW_PROJECT} v{ROBOFLOW_MODEL_VERSION}")

In [ ]:
def run_inference(model, image_path, confidence=0.5):
    """Run segmentation inference on a single image."""
    prediction = model.predict(image_path, confidence=int(confidence * 100))

    return prediction.json()


def visualize_predictions(image_bgr, result_json):
    """Visualize segmentation results on the image."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    display = image_rgb.copy()
    detections = result_json.get("predictions", [])

    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    centroids = []

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image_rgb)

    for i, det in enumerate(detections):
        class_name = det['class']
        confidence = det['confidence']
        color = colors[i % len(colors)]

        # Draw segmentation mask
        if 'points' in det:
            pts = np.array([(p['x'], p['y']) for p in det['points']])
            polygon = Polygon(pts, closed=True)
            p = PatchCollection([polygon], alpha=0.3, facecolors=[color], edgecolors=[color], linewidths=2)
            ax.add_collection(p)

            # Compute centroid
            M = cv2.moments(pts.astype(np.int32))
            if M['m00'] > 0:
                cx = M['m10'] / M['m00']
                cy = M['m01'] / M['m00']
            else:
                cx, cy = det['x'], det['y']
        else:
            cx, cy = det['x'], det['y']

        # Mark centroid
        ax.plot(cx, cy, 'r+', markersize=15, markeredgewidth=3)
        ax.annotate(
            f"{class_name} ({confidence:.2f})",
            (cx, cy), textcoords="offset points", xytext=(10, -15),
            fontsize=10, color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color[:3], alpha=0.8)
        )

        centroids.append({
            'class': class_name,
            'confidence': confidence,
            'centroid_px': (round(cx, 1), round(cy, 1))
        })

    ax.set_title(f"Segmentation Results — {len(detections)} object(s) detected")
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return centroids

print(f"Registered functions.")

In [ ]:
# Run inference on a saved image
# Change this to one of your captured images
TEST_IMAGE =r".\captured_images\img_0000.png"
# os.path.join(OUTPUT_DIR, image_files[0]) if image_files else "test.jpg"

print(f"Running inference on: {TEST_IMAGE}")
image = cv2.imread(TEST_IMAGE)
if image is None:
    print(f"ERROR: Cannot read {TEST_IMAGE}")
else:
    result = run_inference(model, TEST_IMAGE, confidence=CONFIDENCE_THRESHOLD)
    centroids = visualize_predictions(image, result)

    print("\nDetected objects:")
    for c in centroids:
        print(f"  {c['class']}: confidence={c['confidence']:.2f}, center={c['centroid_px']} px")

In [ ]:
# ============================================================
# EXPERIMENT: Try different confidence thresholds
# ============================================================
thresholds = [0.2, 0.5, 0.8]

if image is not None:
    fig, axes = plt.subplots(1, len(thresholds), figsize=(18, 6))
    for ax, thresh in zip(axes, thresholds):
        result = run_inference(model, TEST_IMAGE, confidence=thresh)
        n_det = len(result.get('predictions', []))

        # Simple visualization: just bounding boxes
        disp = cv2.cvtColor(image.copy(), cv2.COLOR_BGR2RGB)
        for det in result.get('predictions', []):
            x1 = int(det['x'] - det['width'] / 2)
            y1 = int(det['y'] - det['height'] / 2)
            x2 = int(det['x'] + det['width'] / 2)
            y2 = int(det['y'] + det['height'] / 2)
            cv2.rectangle(disp, (x1, y1), (x2, y2), (0, 255, 0), 2)

        ax.imshow(disp)
        ax.set_title(f"Threshold: {thresh}\n{n_det} detection(s)")
        ax.axis('off')
    plt.suptitle("Effect of Confidence Threshold", fontsize=14)
    plt.tight_layout()
    plt.show()

---
## 4. ArUco Calibration

The camera sees **pixels**. The robot works in **millimeters**.

We use an ArUco marker of known physical size to compute a **mm-per-pixel** scaling factor.

### What you need:
1. A printed ArUco marker (DICT_5X5_50)
2. The physical size of the marker measured with a ruler or caliper
3. The marker placed flat on the workspace, visible in the camera

In [ ]:
# ============================================================
# TODO: Enter your measured ArUco marker size in mm
# Measure the black square edge-to-edge with a ruler or caliper
# ============================================================
MARKER_SIZE_MM = 20.0  # <-- CHANGE THIS to your actual measurement
# ============================================================

print(f"Marker side size: {MARKER_SIZE_MM} mm")

In [ ]:
# Capture an image with the ArUco marker in view and compute mm_per_pixel
print("Place the ArUco marker flat on the workspace in view of the camera.")
print("Capturing image...\n")

calib_frame = capture_single_frame(use_webcam=USE_WEBCAM)

# Detect the ArUco marker and compute the scale factor
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_50)
detector = cv2.aruco.ArucoDetector(aruco_dict, cv2.aruco.DetectorParameters())
corners, ids, _ = detector.detectMarkers(calib_frame)

mm_per_pixel = None  # Reset before computing
c0_px = None         # Pixel position of ArUco corner C0 (the work object origin)

if ids is None or len(ids) == 0:
    print("ERROR: No ArUco marker detected in the image.")
    print("Check: Is the marker fully visible, flat, and well-lit?")
    print("Check: Are you using a DICT_5X5_50 marker?")
    print("       Generate one at: https://chev.me/arucogen/")
else:
    # Measure the side length in pixels from corner 0 to corner 1
    c = corners[0][0]
    side_px = float(np.linalg.norm(c[0] - c[1]))
    mm_per_pixel = MARKER_SIZE_MM / side_px

    # C0 is corner 0 of the first detected marker — this is the work object origin
    c0_px = c[0]  # (x, y) in pixels

    print(f"Marker detected — ID: {ids[0][0]}")
    print(f"Marker side: {side_px:.1f} px  |  Physical size: {MARKER_SIZE_MM} mm")
    print(f"Scale: {mm_per_pixel:.4f} mm/pixel")
    print(f"Work object origin C0: pixel ({c0_px[0]:.1f}, {c0_px[1]:.1f})")

    # Draw detected marker outline
    disp = cv2.cvtColor(calib_frame.copy(), cv2.COLOR_BGR2RGB)
    cv2.aruco.drawDetectedMarkers(disp, corners, ids)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(disp)

    # Axis arrow length: 1.5× the marker side
    axis_len = side_px * 1.5

    # X axis: C0 → C1 direction
    x_dir = c[1] - c[0]
    x_dir = x_dir / np.linalg.norm(x_dir) * axis_len
    ax.annotate('', xy=(c[0][0] + x_dir[0], c[0][1] + x_dir[1]),
                xytext=(c[0][0], c[0][1]),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
    ax.text(c[0][0] + x_dir[0] + 8, c[0][1] + x_dir[1],
            'X', color='red', fontsize=13, fontweight='bold')

    # Y axis: C0 → C3 direction
    y_dir = c[3] - c[0]
    y_dir = y_dir / np.linalg.norm(y_dir) * axis_len
    ax.annotate('', xy=(c[0][0] + y_dir[0], c[0][1] + y_dir[1]),
                xytext=(c[0][0], c[0][1]),
                arrowprops=dict(arrowstyle='->', color='lime', lw=2.5))
    ax.text(c[0][0] + y_dir[0], c[0][1] + y_dir[1] + 16,
            'Y', color='lime', fontsize=13, fontweight='bold')

    # Mark the origin C0
    ax.plot(c[0][0], c[0][1], 'o', color='yellow', markersize=10, zorder=5,
            label=f"C0 origin ({c[0][0]:.0f}, {c[0][1]:.0f}) px")
    ax.legend(fontsize=10, loc='upper right')

    ax.set_title(
        f"Calibration OK — {mm_per_pixel:.4f} mm/pixel  |  C0 = ({c0_px[0]:.0f}, {c0_px[1]:.0f}) px",
        fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


### Verification — Compare computed distance to ruler measurement

Now check that the calibration is accurate by measuring a real distance:

1. **Keep the ArUco marker in place** (same position as during calibration)
2. **Place one of your trained objects** on the workspace, next to the marker
3. **Measure with a ruler** the distance from **corner C0** of the ArUco marker to the **centre of the object** (in mm)
4. Enter your measurement in `MANUAL_MEASUREMENT_MM` below and run the cell

The program will detect the same distance using only pixels + calibration, and compare the two values.

In [ ]:
# ============================================================
# TODO: Enter your ruler measurement (mm)
# Measure from ArUco corner C0 to the centre of the object
# ============================================================
MANUAL_MEASUREMENT_MM = 130  # <-- CHANGE THIS before running
# ============================================================
 

if mm_per_pixel is None:
    print("Run the calibration cell first.")
else:
    # --- Step 1: Capture new image with object + marker visible ---
    print("Capturing verification image...")
    print("Make sure BOTH the ArUco marker AND the object are in view.\n")
    verify_frame = capture_single_frame(use_webcam=USE_WEBCAM)

    # --- Step 2: Detect ArUco marker → get C0 (corner 0) ---
    aruco_dict_v = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_50)
    detector_v = cv2.aruco.ArucoDetector(aruco_dict_v, cv2.aruco.DetectorParameters())
    corners_v, ids_v, _ = detector_v.detectMarkers(verify_frame)

    if ids_v is None or len(ids_v) == 0:
        print("ERROR: ArUco marker not detected.")
        print("Check: Is the marker flat, well-lit, and fully in frame?")
    else:
        c0 = corners_v[0][0][0]   # Corner 0 of the first detected marker
        print(f"ArUco C0 found at pixel ({c0[0]:.1f}, {c0[1]:.1f})")

        # --- Step 3: Run inference → get object centroids ---
        import tempfile
        tmp_path = os.path.join(tempfile.gettempdir(), "_verify_frame.jpg")
        cv2.imwrite(tmp_path, verify_frame)
        verify_result = run_inference(model, tmp_path, confidence=CONFIDENCE_THRESHOLD)
        preds = verify_result.get("predictions", [])

        if not preds:
            print("ERROR: No objects detected.")
            print("Check: Is the object in view and in your trained classes?")
        else:
            # Use the highest-confidence detection for ruler verification
            best = max(preds, key=lambda p: p['confidence'])
            obj_cx, obj_cy = best['x'], best['y']
            obj_class = best['class']
            obj_conf = best['confidence']
            print(f"Object detected: '{obj_class}' (conf={obj_conf:.2f}) "
                  f"at pixel ({obj_cx:.1f}, {obj_cy:.1f})")

            # --- Step 4: Pixel distance C0 → centroid, convert to mm ---
            dx_px = obj_cx - c0[0]
            dy_px = obj_cy - c0[1]
            dist_px = math.sqrt(dx_px**2 + dy_px**2)
            dist_mm_computed = dist_px * mm_per_pixel

            # --- Step 5: Visualise ---
            display = cv2.cvtColor(verify_frame.copy(), cv2.COLOR_BGR2RGB)
            cv2.aruco.drawDetectedMarkers(display, corners_v, ids_v)

            fig, ax = plt.subplots(figsize=(12, 8))
            ax.imshow(display)
            ax.plot(c0[0], c0[1], 'go', markersize=12,
                    label=f"C0  ({c0[0]:.0f}, {c0[1]:.0f}) px")
            ax.plot(obj_cx, obj_cy, 'r+', markersize=20, markeredgewidth=3,
                    label=f"'{obj_class}' centroid  ({obj_cx:.0f}, {obj_cy:.0f}) px")
            ax.plot([c0[0], obj_cx], [c0[1], obj_cy], 'y--', linewidth=2,
                    label=f"Computed distance = {dist_mm_computed:.1f} mm")
            ax.legend(fontsize=11, loc='upper right')
            ax.set_title(f"Verification: C0 → '{obj_class}' centroid", fontsize=13)
            ax.axis('off')
            plt.tight_layout()
            plt.show()

            # --- Step 6: Compare with manual measurement ---
            print()
            print("=" * 45)
            print(f"  Computed distance:  {dist_mm_computed:.1f} mm")
            print(f"  Manual measurement: {MANUAL_MEASUREMENT_MM:.1f} mm")
            if MANUAL_MEASUREMENT_MM > 0:
                error_mm = abs(dist_mm_computed - MANUAL_MEASUREMENT_MM)
                error_pct = error_mm / MANUAL_MEASUREMENT_MM * 100
                print(f"  Error:              {error_mm:.1f} mm  ({error_pct:.1f}%)")
                print("=" * 45)
                if error_pct < 5:
                    print("  Excellent! Calibration error < 5%")
                elif error_pct < 10:
                    print("  Good. Calibration error < 10%")
                else:
                    print("  Large error — recheck marker size or camera tilt.")
            else:
                print()
                print("  Set MANUAL_MEASUREMENT_MM to your ruler reading above.")
            print("=" * 45)

            # --- Step 7: Update centroids from this image for pick target computation ---
            centroids = []
            for det in preds:
                if 'points' in det:
                    pts = np.array([(p['x'], p['y']) for p in det['points']])
                    M = cv2.moments(pts.astype(np.int32))
                    cx = M['m10'] / M['m00'] if M['m00'] > 0 else det['x']
                    cy = M['m01'] / M['m00'] if M['m00'] > 0 else det['y']
                else:
                    cx, cy = det['x'], det['y']
                centroids.append({
                    'class': det['class'],
                    'confidence': det['confidence'],
                    'centroid_px': (round(cx, 1), round(cy, 1))
                })
            print(f"\nCentroids updated: {len(centroids)} object(s) — run the next cell to compute pick targets.")

        # Cleanup temp file
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


---
## 5. Work object calibration
The work object (coordinate frame) is used so the robot knows how to translate what the camera sees (ArUco marker pose) into its own motion system. By createing the work object at the same location as the coorcinate system for the ArUco marker, the relation between the camera frame and TCP frame can be linked.

## What You Will Do in this Session
1. Calibrate the work object for the robot
2. Draw segmentation masks around each object
3. Train an AI model to recognize your objects
4. Evaluate how well the model works

---
## 6. Pick Point Computation

Combine inference + calibration to compute pick points in robot coordinates.

In [ ]:
# ============================================================
# TODO: Set the pick Z height in the work object frame
# Z=0 is the surface of the workspace (same plane as the ArUco marker).
# Positive Z moves UP. Set PICK_Z to the height the suction cup
# needs to reach down to grab the object (negative = below marker plane).
# ============================================================
PICK_Z = 5.0   # <-- CHANGE THIS (mm, in work object frame)
# ============================================================

print(f"Pick Z: {PICK_Z} mm (work object frame)")


In [ ]:
def compute_pick_targets(centroids, mm_per_pixel, c0_px, pick_z=0.0):
    """Convert pixel centroids to work object coordinates.

    The ArUco marker corner C0 is the origin (0, 0, 0) of the work object.
    Each detection centroid is translated relative to C0 and scaled by
    mm_per_pixel to produce (X, Y) in the robot work object frame.
    """
    if mm_per_pixel is None or mm_per_pixel <= 0:
        print("ERROR: Invalid mm_per_pixel. Run calibration first.")
        return []
    if c0_px is None:
        print("ERROR: c0_px not set. Run calibration first.")
        return []

    c0_x, c0_y = float(c0_px[0]), float(c0_px[1])

    targets = []
    print(f"Work object origin C0: pixel ({c0_x:.1f}, {c0_y:.1f})")
    print(f"Scale: {mm_per_pixel:.4f} mm/pixel\n")
    print(f"{'Class':<12} {'Conf':>5} {'Pixel (x,y)':>18} {'Work obj (x,y,z)':>22}")
    print("-" * 63)

    for c in centroids:
        px_x, px_y = c['centroid_px']

        # Translate relative to C0, then scale to mm.
        wobj_x = abs(px_x - c0_x) * mm_per_pixel
        wobj_y = abs(px_y - c0_y) * mm_per_pixel ## tvekasmt, kan inte gå negativa coordnater men det kan ju gå, så vad är problemet? kanske att det inte är absolutvärde?
        wobj_z = pick_z

        target = {
            'class': c['class'],
            'confidence': c['confidence'],
            'centroid_px': c['centroid_px'],
            'robot_target': (round(wobj_x, 1), round(wobj_y, 1), round(wobj_z, 1)),
        }
        targets.append(target)

        print(f"{c['class']:<12} {c['confidence']:>5.2f} "
              f"({px_x:>6.1f}, {px_y:>6.1f})  "
              f"({wobj_x:>7.1f}, {wobj_y:>7.1f}, {wobj_z:>5.1f})")

    return targets


# ── Capture a fresh image and run inference ───────────────────
if mm_per_pixel is None or c0_px is None:
    print("Run the calibration cell first.")
    pick_targets = []
else:
    print("Capturing fresh image for pick target computation...")
    _pick_frame = capture_single_frame(use_webcam=USE_WEBCAM)
    import tempfile as _tempfile
    _pick_tmp = os.path.join(_tempfile.gettempdir(), "_pick_target_frame.jpg")
    cv2.imwrite(_pick_tmp, _pick_frame)

    _pick_result = run_inference(model, _pick_tmp, confidence=CONFIDENCE_THRESHOLD)
    centroids = visualize_predictions(_pick_frame, _pick_result)

    if os.path.exists(_pick_tmp):
        os.remove(_pick_tmp)

    print(f"\nPick Target Computation (work object frame, C0 = origin)")
    print("=" * 63)
    pick_targets = compute_pick_targets(centroids, mm_per_pixel, c0_px, PICK_Z)


---
## 7. Robot Pick and Place
**IMPORTANT:** Only run this section when:
- The Dobot E6 is powered on and homed
- The teacher has approved your calibration
- The workspace is clear and safe
- Emergency stop is within reach

In [ ]:
# ============================================================
# Robot connection settings — set by teacher
# ============================================================
#ROBOT_IP = "192.168.201.1"     # <-- Your robot's IP on wifi
ROBOT_IP = "192.168.5.1"     # <-- Your robot's IP on ethernet
SAFE_Z = 100.0               # Safe travel height (mm)
DROP_POSITION = (50.0, 200.0, 50.0)  # (X, Y, Z) drop zone
ROBOT_SPEED = 30             # % of max speed
USER_COORD_INDEX = 3         # Work object index defined in robot controller
TOOL_INDEX = 2               # Tool index defined in robot controller
# ============================================================
import sys, os, importlib

_scripts_dir = r"..\scripts"
if _scripts_dir not in sys.path:
    sys.path.insert(0, _scripts_dir)

# Clean up old robot objects before setting up new.
#try:
#    robot.disconnect()
#except Exception:
#    pass
#del robot

# Import robot control module
try:
    import dobot_pick
    importlib.reload(dobot_pick)
    from dobot_pick import DobotE6, pick_object
    print("Robot module loaded.")
except ImportError as e:
    print(f"dobot_pick module not found: {e}")
    print("You can still review the pick targets above without the robot.")


In [ ]:
# Connect to the robot
# Safe to re-run — skips connection if already connected.

if 'robot' in dir() and robot.sock is not None:
    print("Robot already connected — skipping connect.")
    print(f"Robot mode: {robot.get_mode()}")
else:
    robot = DobotE6(ip=ROBOT_IP)
    robot.connect()
    robot.clear_error()
    robot.enable()
    robot.set_speed(ROBOT_SPEED)
    print(f"\nRobot mode: {robot.get_mode()}")  # Should be 5 (idle/enabled)
    print("Robot ready.")


In [ ]:
# ============================================================
# Step 3a — Capture gripper orientation (RX, RY, RZ)
# ============================================================
# Before running this cell:
#   1. Using the robot pendant or DobotStudio, jog the robot so
#      the suction gripper is pointing straight down toward the workspace.
#   2. Visually confirm the orientation looks correct.
#   3. Then run this cell — it reads the current TCP angles and
#      stores them as RX, RY, RZ for all subsequent moves.
#
# Re-run this cell any time you change the gripper orientation.
# ============================================================

print("Reading current TCP pose from robot...")
pose_resp = robot.send("GetPose()")
print(f"Raw GetPose response: {pose_resp}\n")

# V4 response format: 0,{X,Y,Z,Rx,Ry,Rz,J1,J2,J3,J4,J5,J6},GetPose();
try:
    inner = pose_resp.split("{")[1].split("}")[0]
    vals = [float(v.strip()) for v in inner.split(",")]
    # Index: 0=X, 1=Y, 2=Z, 3=Rx, 4=Ry, 5=Rz
    RX, RY, RZ = vals[3], vals[4], vals[5]
    print("Gripper orientation captured successfully:")
    print(f"  RX = {RX:.4f} deg")
    print(f"  RY = {RY:.4f} deg")
    print(f"  RZ = {RZ:.4f} deg")
    print("\nThese values will be used for all moves in this session.")
    print("Re-run this cell if you physically re-orient the gripper.")
except (IndexError, ValueError) as e:
    print(f"ERROR parsing GetPose response: {e}")
    print("Set RX, RY, RZ manually below and re-run.")
    RX = 177.9350
    RY =   3.3567
    RZ =  62.6958
    print(f"Using fallback: RX={RX}, RY={RY}, RZ={RZ}")


In [ ]:
# ============================================================
# Step 3b — Activate work object and verify origin
# ============================================================
# USER_COORD_INDEX is set in the settings cell above.
# Send a move to (0, 0, VERIFY_Z) — the robot should be directly
# above ArUco marker corner C0 if the work object is correct.
# ============================================================
VERIFY_Z = 5.0        # Safe height above the origin point (mm)

# Activate the user coordinate system
resp = robot.send(f"User({USER_COORD_INDEX})")
print(f"User({USER_COORD_INDEX}) → {resp}")

resp = robot.send(f"Tool({TOOL_INDEX})")
print(f"Tool({TOOL_INDEX}) → {resp}")

# Read current pose before moving
resp_pose = robot.send("GetPose()")
print(f"Pose before move: {resp_pose}")

print(f"\nMoving to work object origin (0, 0, {VERIFY_Z}) ...")
print("MAKE SURE THE WORKSPACE IS CLEAR.\n")

robot.move_to(0.0, 0.0, VERIFY_Z, rx=RX, ry=RY, rz=RZ,
              user=USER_COORD_INDEX, tool=TOOL_INDEX)

resp_pose2 = robot.send("GetPose()")
print(f"Pose after move:  {resp_pose2}")
print()
print("If the TCP is directly above ArUco corner C0, the work object is correct.")
print("If not, adjust USER_COORD_INDEX or redefine the work object in the controller.")

robot.move_to(48.5836,-17.2863,257.6433, rx=RX, ry=RY, rz=RZ,user=USER_COORD_INDEX, tool=TOOL_INDEX)

print("back to home")



In [ ]:
# ============================================================
# Step 3c — Test move: go to work-object origin at safe height
#
# RX, RY, RZ are captured automatically in Step 1b above.
# Run Step 1b first if you see a NameError for RX/RY/RZ.
# ============================================================
TARGET_X = 0.0    # work-object X (ArUco marker origin)
TARGET_Y = 0.0    # work-object Y
TARGET_Z = 250.0  # safe height above workspace (mm)

print(f"Using orientation: RX={RX:.4f}  RY={RY:.4f}  RZ={RZ:.4f}")
print(f"Pose before move: {robot.get_pose()}")
print()
print(f"Sending: MovL(pose={{{TARGET_X},{TARGET_Y},{TARGET_Z},{RX},{RY},{RZ}}},user={USER_COORD_INDEX},tool={TOOL_INDEX})")
print("MAKE SURE WORKSPACE IS CLEAR!\n")

robot.move_to(TARGET_X, TARGET_Y, TARGET_Z, rx=RX, ry=RY, rz=RZ,
              user=USER_COORD_INDEX, tool=TOOL_INDEX)

print()
print(f"Pose after move: {robot.get_pose()}")
print()
print("If the TCP is directly above ArUco corner C0, the work object is correct.")


In [ ]:

# ============================================================
# Step 5 — Capture → Infer → Pick the best detected object
#
# Only run after:
#   - Step 2 verified the work object is correct
#   - mm_per_pixel and c0_px are set (calibration done)
# ============================================================
X_OFFSET =  0.0   # <-- Manual X adjustment (mm). Positive = further in X direction.
Y_OFFSET =  0.0   # <-- Manual Y adjustment (mm). Positive = further in Y direction.
Z_OFFSET =  0.0   # <-- Manual Z adjustment (mm). Positive = higher, negative = lower.
                   #     Tune these if the pick position is consistently off.

# ── Step 3a: Move to home so robot is out of the camera view ─
robot.move_to(48.5836, -17.2863, 257.6433, rx=RX, ry=RY, rz=RZ,
              user=USER_COORD_INDEX, tool=TOOL_INDEX)
print("Robot at home — capturing fresh image...")

# ── Step 3b: Capture a new image ─────────────────────────────
import tempfile as _tempfile
_step3_frame = capture_single_frame(use_webcam=USE_WEBCAM)
_step3_tmp = os.path.join(_tempfile.gettempdir(), "_step3_frame.jpg")
cv2.imwrite(_step3_tmp, _step3_frame)

# ── Step 3c: Run inference ────────────────────────────────────
print("Running inference...")
_step3_result = run_inference(model, _step3_tmp, confidence=CONFIDENCE_THRESHOLD)
centroids = visualize_predictions(_step3_frame, _step3_result)

if os.path.exists(_step3_tmp):
    os.remove(_step3_tmp)

# ── Step 3d: Compute pick targets ────────────────────────────
print(f"\nPick Target Computation (work object frame, C0 = origin)")
print("=" * 63)
pick_targets = compute_pick_targets(centroids, mm_per_pixel, c0_px, PICK_Z)

# ── Step 3e: Execute the pick ─────────────────────────────────
if not pick_targets:
    print("No pick targets detected — skipping pick.")
else:
    best = pick_targets[0]   # highest confidence
    tx, ty, tz = best['robot_target']
    tx_adjusted = tx + X_OFFSET
    ty_adjusted = ty + Y_OFFSET
    tz_adjusted = tz + Z_OFFSET

    print(f"\nPicking: {best['class']}  conf={best['confidence']:.2f}")
    print(f"Target : ({tx:.1f}, {ty:.1f}, {tz:.1f}) mm")
    print(f"Offsets: X={X_OFFSET:+.1f}  Y={Y_OFFSET:+.1f}  Z={Z_OFFSET:+.1f}")
    print(f"Sending: ({tx_adjusted:.1f}, {ty_adjusted:.1f}, {tz_adjusted:.1f}) mm")
    print("MAKE SURE WORKSPACE IS CLEAR!\n")

    pick_object(robot, tx_adjusted, ty_adjusted, tz_adjusted,
                rx=RX, ry=RY, rz=RZ,
                user=USER_COORD_INDEX, tool=TOOL_INDEX)
    print("Done.")

    robot.move_to(48.5836, -17.2863, 257.6433, rx=RX, ry=RY, rz=RZ,
                  user=USER_COORD_INDEX, tool=TOOL_INDEX)

print("back to home")


In [ ]:
# Disconnect from the robot when done
robot.disconnect()
print("Robot disconnected.")

---
## 8. Full Pipeline — Capture → Segment → Pick

This cell runs the complete pipeline in sequence:
1. Capture a new image
2. Run segmentation
3. Compute pick targets
4. Execute the pick (if robot is connected)

**Only run this after all previous sections are working correctly.**

In [ ]:
# Run full pipeline
def full_pipeline(model, mm_per_pixel, c0_px, pick_z=0.0, robot=None,
                  rx=0.0, ry=0.0, rz=0.0, user=None, tool=None,
                  use_webcam=True, confidence=0.5,
                  x_offset=0.0, y_offset=0.0, z_offset=0.0):
    """Run the complete vision-to-pick pipeline."""
    print("=" * 60)
    print("FULL PIPELINE")
    print("=" * 60)

    # Step 1: Capture
    print("\n[1/4] Capturing image...")
    robot.move_to(48.5836,-17.2863,257.6433, rx=RX, ry=RY, rz=RZ,user=USER_COORD_INDEX, tool=TOOL_INDEX)
    
    frame = capture_single_frame(use_webcam=use_webcam)
    temp_path = "_pipeline_frame.jpg"
    cv2.imwrite(temp_path, frame)
    print(f"  Image captured: {frame.shape[1]}x{frame.shape[0]}")

    # Step 2: Inference
    print("\n[2/4] Running segmentation...")
    result = run_inference(model, temp_path, confidence=confidence)
    centroids = visualize_predictions(frame, result)
    print(f"  Found {len(centroids)} object(s)")

    if not centroids:
        print("\nNo objects detected. Pipeline stopped.")
        if os.path.exists(temp_path):
            os.remove(temp_path)
        return []

    # Step 3: Pick targets
    print("\n[3/4] Computing pick targets...")
    targets = compute_pick_targets(centroids, mm_per_pixel, c0_px, pick_z)

    # Step 4: Robot pick
    if robot is not None and targets:
        best = targets[0]
        tx, ty, tz = best['robot_target']   # unpack from this run's detections
        tx_adj = abs(tx) + x_offset
        ty_adj = abs(ty) + y_offset
        tz_adj = tz + z_offset
        print(f"\n[4/4] Picking {best['class']} at ({tx}, {ty}, {tz})...")
        if x_offset or y_offset or z_offset:
            print(f"  Offsets applied: X={x_offset:+.1f}  Y={y_offset:+.1f}  Z={z_offset:+.1f}")
            print(f"  Adjusted target: ({tx_adj:.1f}, {ty_adj:.1f}, {tz_adj:.1f})")
        try:
            pick_object(robot, tx_adj, ty_adj, tz_adj,
                        rx=rx, ry=ry, rz=rz,
                        user=user, tool=tool)
            print("  Pick successful!")

            robot.move_to(48.5836,-17.2863,257.6433, rx=RX, ry=RY, rz=RZ,user=USER_COORD_INDEX, tool=TOOL_INDEX)
            print("Done.")
        except (ValueError, RuntimeError) as e:
            print(f"  ERROR: {e}")
            if "socket" in str(e).lower() or "connect" in str(e).lower():
                print("  --> Re-run the robot connect cell then try again.")
    else:
        print("\n[4/4] Robot not connected — skipping pick execution.")
        print("  Pick targets are ready for when the robot is connected.")

    # Cleanup
    if os.path.exists(temp_path):
        os.remove(temp_path)

    print("\n" + "=" * 60)
    return targets

print("PIPELINE COMPLETE")
print("=" * 60)


In [ ]:
# Automated pipeline
# Set robot=robot if the robot is connected, or robot=None for vision-only mode

for i in range(1): 
  pipeline_targets = full_pipeline(
    model=model,
    mm_per_pixel=mm_per_pixel,
    c0_px=c0_px,
    pick_z=PICK_Z,
    robot=robot,              # Set to None for vision-only mode (no robot moves)
    rx=RX, ry=RY, rz=RZ,
    user=USER_COORD_INDEX, tool=TOOL_INDEX,
    use_webcam=USE_WEBCAM,
    confidence=CONFIDENCE_THRESHOLD,
    x_offset=X_OFFSET, y_offset=Y_OFFSET, z_offset=Z_OFFSET,
)


---
## 9. Reflection Questions

Answer these after completing the lab:

1. **What was the biggest source of error in your picks?**
   - _Your answer:_

2. **How accurate was your calibration?**
   - _Your answer:_

3. **What would you change to improve reliability?**
   - _Your answer:_

4. **Could this system work in a real factory? What would need to change?**
   - _Your answer:_

5. **What did you learn that surprised you?**
   - _Your answer:_